In [1]:
!pip install yfinance pandas numpy statsmodels


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import itertools


In [3]:
candidate_tickers = [
    'BTC-USD','ETH-USD','XRP-USD','BNB-USD','SOL-USD','DOGE-USD','ADA-USD','TRX-USD',
    'AVAX-USD','SHIB-USD','DOT-USD','LINK-USD','BCH-USD','NEAR-USD','LTC-USD','MATIC-USD',
    'UNI-USD','ICP-USD','ETC-USD','XLM-USD','APT-USD','FIL-USD','HBAR-USD','ARB-USD',
    'VET-USD','OP-USD','MKR-USD','INJ-USD','GRT-USD','AAVE-USD','ALGO-USD','QNT-USD',
    'EGLD-USD','SAND-USD','MANA-USD','THETA-USD','AXS-USD','XTZ-USD','EOS-USD','FLOW-USD',
    'CHZ-USD','KAVA-USD','RUNE-USD','ZEC-USD','DASH-USD','XMR-USD','NEO-USD','CRV-USD',
    'SNX-USD','COMP-USD',
]

# Fixed window, can adjust date to whenever. sample four year period
START_DATE = '2022-09-17'
END_DATE = '2026-09-17'

raw_daily_native = yf.download(candidate_tickers, start=START_DATE, end=END_DATE, interval='1d', progress=False)
closes_dn_raw = raw_daily_native['Close']
vols_dn_raw = raw_daily_native['Volume']
highs_dn_raw = raw_daily_native['High']
lows_dn_raw = raw_daily_native['Low']

# Point-in-time universe construction: a coin is included from whenever it actually has price
# data, rather than being filtered upfront by its completeness over the FULL four-year window.
# Old method committed survivorship bias
completely_missing = closes_dn_raw.columns[closes_dn_raw.isna().all()].tolist()
kept_tickers_dn = sorted([t for t in candidate_tickers if t not in completely_missing])
dropped_dn = completely_missing

print(f"Data window: {START_DATE} to {END_DATE}")
print(f"Kept {len(kept_tickers_dn)} / {len(candidate_tickers)} tickers (dropped only complete download failures).")
if dropped_dn:
    print("Dropped (no data returned at all):", dropped_dn)

prices_dn = closes_dn_raw[kept_tickers_dn].dropna(how='all')
volumes_dn = vols_dn_raw[kept_tickers_dn].reindex(prices_dn.index).fillna(0)
highs_dn = highs_dn_raw[kept_tickers_dn].reindex(prices_dn.index)
lows_dn = lows_dn_raw[kept_tickers_dn].reindex(prices_dn.index)
log_returns_dn = np.log(prices_dn / prices_dn.shift(1)).dropna(how='all')
volumes_dn = volumes_dn.loc[log_returns_dn.index]
highs_dn = highs_dn.loc[log_returns_dn.index]
lows_dn = lows_dn.loc[log_returns_dn.index]
prices_dn = prices_dn.loc[log_returns_dn.index]

print(f"Native daily universe: {log_returns_dn.shape[1]} coins, {log_returns_dn.shape[0]} daily bars "
      f"({log_returns_dn.shape[0]/365:.1f} years).")

# Transparency: coin count now legitimately varies over time (coins enter as they're listed,
# rather than being excluded from the whole study by a full-sample filter).
active_coin_counts = prices_dn.notna().sum(axis=1)
print(f"\nCoins with live data: {active_coin_counts.iloc[0]} on day 1, "
      f"{active_coin_counts.iloc[-1]} on the last day, minimum {active_coin_counts.min()} "
      f"(on {active_coin_counts.idxmin().date()}).")

late_starters = [(t, prices_dn[t].first_valid_index().date()) for t in kept_tickers_dn
                  if prices_dn[t].first_valid_index() is not None
                  and (prices_dn[t].first_valid_index() - prices_dn.index[0]).days > 30]
if late_starters:
    print("\nCoins now correctly included only from their actual listing date "
          "(previously excluded from the whole study by the full-sample filter):")
    for t, d in late_starters:
        print(f"  {t}: first data {d}")


$GRT-USD: No data found, symbol may be delisted
$COMP-USD: No data found, symbol may be delisted
$UNI-USD: No data found, symbol may be delisted
$APT-USD: No data found, symbol may be delisted
$MATIC-USD: No data found, symbol may be delisted

5 Failed downloads:
['GRT-USD', 'COMP-USD', 'UNI-USD', 'APT-USD', 'MATIC-USD']: No data found, symbol may be delisted


Data window: 2022-09-17 to 2026-09-17
Kept 45 / 50 tickers (dropped only complete download failures).
Dropped (no data returned at all): ['APT-USD', 'COMP-USD', 'GRT-USD', 'MATIC-USD', 'UNI-USD']
Native daily universe: 45 coins, 1460 daily bars (4.0 years).

Coins with live data: 45 on day 1, 45 on the last day, minimum 45 (on 2022-09-18).


In [4]:
def run_backtest(prices, volumes, params, highs=None, lows=None, bar_freq='1D',
                  net_cost_bps=0.0020, start_idx=None, end_idx=None, label=""):
    """
    Cross-sectional mean-reversion backtest engine: each coin's return is z-scored against
    the OTHER coins in the basket at that same bar (nets out common market-wide moves),
    combined with a per-coin low-volume filter. Positions are equal-weighted across whatever
    coins are active; transaction cost is charged on actual portfolio weight turnover.

    highs/lows: optional. If given, stop-loss is checked against the bar's High/Low and capped
    at exactly -stop_loss if pierced intrabar.

    Optional regime filter (params keys 'regime_window' + 'regime_z_threshold'): blocks new
    entries when the basket's own rolling trend-strength z-score exceeds the threshold.

    params needs: z_entry, vol_entry, profit_hurdle, stop_loss, max_hold_bars, xs_window, vol_window.

    Returns a dict with n_trades, gross_sharpe, net_sharpe, gross_daily/net_daily return series,
    a full trade_log DataFrame, and regime_blocked_entries.
    """
    prices_bar = prices.resample(bar_freq).last().dropna(how='all')
    volumes_bar = volumes.resample(bar_freq).sum().reindex(prices_bar.index)
    log_returns_bar = np.log(prices_bar / prices_bar.shift(1)).dropna(how='all')
    volumes_bar = volumes_bar.loc[log_returns_bar.index]
    prices_bar = prices_bar.loc[log_returns_bar.index]

    use_intrabar_stop = highs is not None and lows is not None
    if use_intrabar_stop:
        highs_bar = highs.resample(bar_freq).max().reindex(prices_bar.index).loc[log_returns_bar.index]
        lows_bar = lows.resample(bar_freq).min().reindex(prices_bar.index).loc[log_returns_bar.index]

    cum_ret = log_returns_bar.rolling(window=params['xs_window']).sum()
    ret_z = cum_ret.sub(cum_ret.mean(axis=1), axis=0).div(cum_ret.std(axis=1), axis=0)

    vol_mean = volumes_bar.rolling(window=params['vol_window']).mean()
    vol_std = volumes_bar.rolling(window=params['vol_window']).std()
    vol_z = (volumes_bar - vol_mean) / vol_std

    use_regime_filter = 'regime_window' in params and 'regime_z_threshold' in params
    if use_regime_filter:
        market_ret = log_returns_bar.mean(axis=1)
        rw = params['regime_window']
        trend_z = market_ret.rolling(rw).sum() / (market_ret.rolling(rw).std() * np.sqrt(rw))
        regime_allow_bar = trend_z.abs() < params['regime_z_threshold']

    if start_idx is None: start_idx = 0
    if end_idx is None: end_idx = len(log_returns_bar)
    sub_returns = log_returns_bar.iloc[start_idx:end_idx]
    sub_prices = prices_bar.iloc[start_idx:end_idx]
    sub_ret_z = ret_z.iloc[start_idx:end_idx]
    sub_vol_z = vol_z.iloc[start_idx:end_idx]
    if use_intrabar_stop:
        sub_highs = highs_bar.iloc[start_idx:end_idx]
        sub_lows = lows_bar.iloc[start_idx:end_idx]
    if use_regime_filter:
        sub_regime_allow = regime_allow_bar.iloc[start_idx:end_idx]

    z_entry, vol_entry = params['z_entry'], params['vol_entry']
    profit_hurdle, stop_loss, max_hold = params['profit_hurdle'], params['stop_loss'], params['max_hold_bars']

    positions = pd.DataFrame(0.0, index=sub_returns.index, columns=sub_returns.columns)
    realized_returns = pd.DataFrame(0.0, index=sub_returns.index, columns=sub_returns.columns)
    trade_log = []
    regime_blocked_entries = 0

    for col in sub_returns.columns:
        col_idx = sub_returns.columns.get_loc(col)
        curr_pos, entry_price, hold_count = 0.0, 0.0, 0
        for t in range(len(sub_returns)):
            z_r = sub_ret_z.iat[t, col_idx]
            z_v = sub_vol_z.iat[t, col_idx]
            curr_price = sub_prices.iat[t, col_idx]
            if np.isnan(z_r) or np.isnan(z_v) or np.isnan(curr_price):
                continue

            if curr_pos != 0.0:
                hold_count += 1
                realized_returns.iat[t, col_idx] = sub_returns.iat[t, col_idx]

                intrabar_stop_hit = False
                if use_intrabar_stop:
                    bar_high = sub_highs.iat[t, col_idx]
                    bar_low = sub_lows.iat[t, col_idx]
                    if curr_pos == 1.0 and not np.isnan(bar_low):
                        intrabar_stop_hit = bar_low <= entry_price * (1 - stop_loss)
                    elif curr_pos == -1.0 and not np.isnan(bar_high):
                        intrabar_stop_hit = bar_high >= entry_price * (1 + stop_loss)

                if intrabar_stop_hit:
                    capped_ret = np.log(1 - stop_loss) if curr_pos == 1.0 else np.log(1 + stop_loss)
                    realized_returns.iat[t, col_idx] = capped_ret
                    trade_log.append({'coin': col, 'gross_pnl': -stop_loss, 'bars_held': hold_count, 'exit_reason': 'stop_intrabar'})
                    curr_pos, entry_price, hold_count = 0.0, 0.0, 0
                else:
                    unrealized_pnl = (curr_price - entry_price) / entry_price if curr_pos == 1.0 else (entry_price - curr_price) / entry_price
                    reverted = (curr_pos == 1.0 and z_r >= 0.0) or (curr_pos == -1.0 and z_r <= 0.0)
                    cleared = unrealized_pnl > profit_hurdle
                    stop_hit_close = unrealized_pnl < -stop_loss
                    time_hit = hold_count >= max_hold
                    if (reverted and cleared) or stop_hit_close or time_hit:
                        reason = 'stop_close' if stop_hit_close else ('profit_reversion' if (reverted and cleared) else 'time')
                        trade_log.append({'coin': col, 'gross_pnl': unrealized_pnl, 'bars_held': hold_count, 'exit_reason': reason})
                        curr_pos, entry_price, hold_count = 0.0, 0.0, 0
            elif curr_pos == 0.0:
                regime_ok = bool(sub_regime_allow.iat[t]) if use_regime_filter else True
                signal_fired = (z_r < -z_entry and z_v < vol_entry) or (z_r > z_entry and z_v < vol_entry)
                if signal_fired and not regime_ok:
                    regime_blocked_entries += 1
                elif z_r < -z_entry and z_v < vol_entry and regime_ok:
                    curr_pos, entry_price, hold_count = 1.0, curr_price, 0
                elif z_r > z_entry and z_v < vol_entry and regime_ok:
                    curr_pos, entry_price, hold_count = -1.0, curr_price, 0

            positions.iat[t, col_idx] = curr_pos

    executed_positions = positions.shift(1).fillna(0)

    # ---- Portfolio construction + turnover-based cost (the reviewer's fix) ----
    active_counts = executed_positions.abs().sum(axis=1)
    safe_divisor = active_counts.replace(0, 1)
    weights = executed_positions.div(safe_divisor, axis=0)

    gross_bar_returns = (weights * realized_returns).sum(axis=1)
    weight_turnover = weights.diff().fillna(0).abs().sum(axis=1)
    net_bar_returns = gross_bar_returns - weight_turnover * net_cost_bps

    raw_trade_diffs = executed_positions.diff().abs().fillna(0)
    n_trades = int(raw_trade_diffs.sum().sum())

    gross_daily = gross_bar_returns.resample('D').sum()
    net_daily = net_bar_returns.resample('D').sum()
    ann_factor = np.sqrt(365)
    gross_sharpe = ann_factor * gross_daily.mean() / gross_daily.std() if gross_daily.std() > 0 else 0.0
    net_sharpe = ann_factor * net_daily.mean() / net_daily.std() if net_daily.std() > 0 else 0.0

    return {
        'label': label, 'total_bars': len(sub_returns), 'n_trades': n_trades,
        'gross_sharpe': round(gross_sharpe, 3), 'net_sharpe': round(net_sharpe, 3),
        'gross_daily': gross_daily, 'net_daily': net_daily,
        'trade_log': pd.DataFrame(trade_log), 'regime_blocked_entries': regime_blocked_entries,
    }


def freeze_params_on_train(prices, volumes, grid, fixed_params, train_end_idx, highs=None, lows=None,
                            bar_freq='1D', min_trades=20, train_start_idx=0):
    """Grid search over `grid` (dict of param-name -> list of values), scored by net Sharpe on
    train only, filtered to combos with >= min_trades."""
    keys = list(grid.keys())
    results = []
    for combo in itertools.product(*grid.values()):
        p = dict(zip(keys, combo), **fixed_params)
        r = run_backtest(prices, volumes, p, highs=highs, lows=lows, bar_freq=bar_freq,
                          start_idx=train_start_idx, end_idx=train_end_idx, label='train')
        results.append((p, r))
    qualified = [(p, r) for p, r in results if r['n_trades'] >= min_trades]
    pool = qualified if qualified else results
    return max(pool, key=lambda pr: pr[1]['net_sharpe'])


def walk_forward_validate(prices, volumes, grid, fixed_params, fold_bounds, highs=None, lows=None,
                           bar_freq='1D', min_trades=20):
    """Re-freezes independently on each fold's train slice, evaluates once on the next slice."""
    rows = []
    for i in range(len(fold_bounds) - 1):
        train_end = fold_bounds[i]
        test_start, test_end = fold_bounds[i], fold_bounds[i + 1]
        p_i, _ = freeze_params_on_train(prices, volumes, grid, fixed_params, train_end_idx=train_end,
                                         highs=highs, lows=lows, bar_freq=bar_freq, min_trades=min_trades)
        test_r_i = run_backtest(prices, volumes, p_i, highs=highs, lows=lows, bar_freq=bar_freq,
                                 start_idx=test_start, end_idx=test_end, label=f'fold{i+1}')
        row = {'Fold': i + 1, 'Test Start': test_r_i['gross_daily'].index[0].date() if len(test_r_i['gross_daily']) else None,
               'Test End': test_r_i['gross_daily'].index[-1].date() if len(test_r_i['gross_daily']) else None,
               **{k: p_i[k] for k in grid.keys()}, 'Test Trades': test_r_i['n_trades'],
               'Test Gross Sharpe': test_r_i['gross_sharpe'], 'Test Net Sharpe': test_r_i['net_sharpe'],
               'Blocked Entries': test_r_i.get('regime_blocked_entries', 0)}
        rows.append(row)
    return pd.DataFrame(rows)


In [5]:
total_bars_daily = len(prices_dn.resample('1D').last().dropna(how='all')) - 1
split_idx_daily = int(total_bars_daily * 0.7)
fold_bounds_daily = [int(total_bars_daily * f) for f in [0.2, 0.4, 0.6, 0.8, 1.0]]
print(f"Total daily bars: {total_bars_daily} (train ~{split_idx_daily}, test ~{total_bars_daily - split_idx_daily})")

grid_daily = {
    'z_entry': [1.5, 2.0, 2.5],
    'vol_entry': [0.3, 0.5],
    'profit_hurdle': [0.01, 0.02, 0.03],
    'stop_loss': [0.03, 0.05, 0.07],
}
fixed_daily = dict(max_hold_bars=5, xs_window=1, vol_window=5)

frozen_daily, train_daily = freeze_params_on_train(prices_dn, volumes_dn, grid_daily, fixed_daily,
                                                    train_end_idx=split_idx_daily, highs=highs_dn, lows=lows_dn,
                                                    bar_freq='1D', min_trades=20)
test_daily = run_backtest(prices_dn, volumes_dn, frozen_daily, highs=highs_dn, lows=lows_dn,
                           bar_freq='1D', start_idx=split_idx_daily, label='test')

print("\nFrozen daily params (selected on train only, corrected cost model):", frozen_daily)
print(pd.DataFrame([
    {'Period': 'Train', 'Trades': train_daily['n_trades'], 'Gross Sharpe': train_daily['gross_sharpe'], 'Net Sharpe (20bps)': train_daily['net_sharpe']},
    {'Period': 'Test', 'Trades': test_daily['n_trades'], 'Gross Sharpe': test_daily['gross_sharpe'], 'Net Sharpe (20bps)': test_daily['net_sharpe']},
]).to_string(index=False))

wf_daily = walk_forward_validate(prices_dn, volumes_dn, grid_daily, fixed_daily, fold_bounds_daily,
                                  highs=highs_dn, lows=lows_dn, bar_freq='1D', min_trades=20)
print("\n5-fold walk-forward, daily bars (corrected cost model):")
print(wf_daily.to_string(index=False))


Total daily bars: 1459 (train ~1021, test ~438)

Frozen daily params (selected on train only, corrected cost model): {'z_entry': 2.0, 'vol_entry': 0.5, 'profit_hurdle': 0.03, 'stop_loss': 0.03, 'max_hold_bars': 5, 'xs_window': 1, 'vol_window': 5}
Period  Trades  Gross Sharpe  Net Sharpe (20bps)
 Train    1689         0.653               0.364
  Test     723         2.462               1.392

5-fold walk-forward, daily bars (corrected cost model):
 Fold Test Start   Test End  z_entry  vol_entry  profit_hurdle  stop_loss  Test Trades  Test Gross Sharpe  Test Net Sharpe  Blocked Entries
    1 2023-07-07 2024-04-23      2.5        0.5           0.03       0.03          247             -1.029           -2.161                0
    2 2024-04-24 2025-02-09      2.0        0.5           0.03       0.03          513              0.401           -1.016                0
    3 2025-02-10 2025-11-28      2.0        0.5           0.03       0.03          512             -0.160           -1.615       

In [6]:
regime_grid = {
    'regime_window': [10, 20, 30, 45, 60, 90],
    'regime_z_threshold': [1.0, 1.5, 2.0, 2.5],
}
fixed_with_regime = {**fixed_daily, 'z_entry': frozen_daily['z_entry'], 'vol_entry': frozen_daily['vol_entry'],
                      'profit_hurdle': frozen_daily['profit_hurdle'], 'stop_loss': frozen_daily['stop_loss']}

frozen_regime, train_regime = freeze_params_on_train(prices_dn, volumes_dn, regime_grid, fixed_with_regime,
                                                      train_end_idx=split_idx_daily, highs=highs_dn, lows=lows_dn,
                                                      bar_freq='1D', min_trades=20)
test_regime = run_backtest(prices_dn, volumes_dn, frozen_regime, highs=highs_dn, lows=lows_dn,
                            bar_freq='1D', start_idx=split_idx_daily, label='test')

print("Regime-filtered params (selected on train only):", frozen_regime)
print(pd.DataFrame([
    {'Period': 'Train', 'Trades': train_regime['n_trades'], 'Blocked': train_regime['regime_blocked_entries'],
     'Gross Sharpe': train_regime['gross_sharpe'], 'Net Sharpe (20bps)': train_regime['net_sharpe']},
    {'Period': 'Test', 'Trades': test_regime['n_trades'], 'Blocked': test_regime['regime_blocked_entries'],
     'Gross Sharpe': test_regime['gross_sharpe'], 'Net Sharpe (20bps)': test_regime['net_sharpe']},
]).to_string(index=False))

wf_regime = walk_forward_validate(prices_dn, volumes_dn, regime_grid, fixed_with_regime, fold_bounds_daily,
                                   highs=highs_dn, lows=lows_dn, bar_freq='1D', min_trades=20)
print("\n5-fold walk-forward, WITH regime filter:")
print(wf_regime.to_string(index=False))
print("\nCompare fold-by-fold to the no-filter walk-forward above -- does the filter reduce the")
print("number/severity of negative folds, or leave the same folds negative regardless?")


Regime-filtered params (selected on train only): {'regime_window': 60, 'regime_z_threshold': 1.0, 'max_hold_bars': 5, 'xs_window': 1, 'vol_window': 5, 'z_entry': 2.0, 'vol_entry': 0.5, 'profit_hurdle': 0.03, 'stop_loss': 0.03}
Period  Trades  Blocked  Gross Sharpe  Net Sharpe (20bps)
 Train    1073      354         0.741               0.555
  Test     485      149         1.374               0.486

5-fold walk-forward, WITH regime filter:
 Fold Test Start   Test End  regime_window  regime_z_threshold  Test Trades  Test Gross Sharpe  Test Net Sharpe  Blocked Entries
    1 2023-07-07 2024-04-23             60                 1.0          277              1.094            1.009              107
    2 2024-04-24 2025-02-09             60                 1.0          364              0.656           -0.718               92
    3 2025-02-10 2025-11-28             60                 1.0          412             -0.045           -1.360               57
    4 2025-11-29 2026-09-16             6

In [7]:
cost_grid_bps = [0, 2, 5, 10, 15, 20, 30, 40]
sens_daily_rows = []
for bps in cost_grid_bps:
    r_tr = run_backtest(prices_dn, volumes_dn, frozen_daily, highs=highs_dn, lows=lows_dn,
                         bar_freq='1D', net_cost_bps=bps/10000, end_idx=split_idx_daily, label='train')
    r_te = run_backtest(prices_dn, volumes_dn, frozen_daily, highs=highs_dn, lows=lows_dn,
                         bar_freq='1D', net_cost_bps=bps/10000, start_idx=split_idx_daily, label='test')
    sens_daily_rows.append({'Cost (bps/trade)': bps, 'Train Net Sharpe': r_tr['net_sharpe'], 'Test Net Sharpe': r_te['net_sharpe']})

print("Net Sharpe across assumed per-trade cost, daily design (corrected turnover-based cost):")
print(pd.DataFrame(sens_daily_rows).to_string(index=False))


Net Sharpe across assumed per-trade cost, daily design (corrected turnover-based cost):
 Cost (bps/trade)  Train Net Sharpe  Test Net Sharpe
                0             0.653            2.462
                2             0.624            2.355
                5             0.581            2.195
               10             0.508            1.927
               15             0.436            1.660
               20             0.364            1.392
               30             0.219            0.857
               40             0.074            0.323


In [8]:
test_daily_log = test_daily['trade_log']
print(f"Round trips (test period, daily bars): {len(test_daily_log)}")

per_coin_daily = test_daily_log.groupby('coin')['gross_pnl'].agg(['count', 'mean', 'sum'])
per_coin_daily['win_rate'] = test_daily_log.groupby('coin')['gross_pnl'].apply(lambda s: (s > 0).mean())
per_coin_daily = per_coin_daily.sort_values('sum')
print("\nPer-coin gross P&L breakdown, sorted worst to best:")
print(per_coin_daily.to_string())

per_coin_full_daily = test_daily_log.groupby('coin')['gross_pnl'].sum().sort_values(ascending=False)
total_pnl_daily = per_coin_full_daily.sum()
total_positive_daily = per_coin_full_daily[per_coin_full_daily > 0].sum()

print(f"\nTotal gross P&L: {total_pnl_daily:.4f}  |  Positive-coin sum: {total_positive_daily:.4f} ({len(per_coin_full_daily[per_coin_full_daily>0])} coins)"
      f"  |  Negative-coin sum: {per_coin_full_daily[per_coin_full_daily<0].sum():.4f} ({len(per_coin_full_daily[per_coin_full_daily<0])} coins)")

print("\nTop 5 coins by gross P&L, and their share of total POSITIVE P&L:")
for coin, pnl in per_coin_full_daily.head(5).items():
    print(f"  {coin:12s} {pnl:+.4f}   ({pnl/total_positive_daily*100:.1f}% of all positive P&L)")

top1_coins = per_coin_full_daily.head(1).index.tolist()
top3_coins = per_coin_full_daily.head(3).index.tolist()
top1_share_daily = per_coin_full_daily.iloc[0] / total_pnl_daily * 100
top3_share_daily = per_coin_full_daily.head(3).sum() / total_pnl_daily * 100
print(f"\nTop 1 coin ({top1_coins[0]}) = {top1_share_daily:.1f}% of TOTAL net gross P&L")
print(f"Top 3 coins ({top3_coins}) = {top3_share_daily:.1f}% of TOTAL net gross P&L")


Round trips (test period, daily bars): 363

Per-coin gross P&L breakdown, sorted worst to best:
           count      mean       sum  win_rate
coin                                          
XMR-USD       24 -0.011175 -0.268204  0.250000
FLOW-USD      27 -0.008075 -0.218027  0.148148
EGLD-USD       8 -0.019513 -0.156107  0.125000
XLM-USD        7 -0.021338 -0.149363  0.142857
TRX-USD        8 -0.015364 -0.122913  0.250000
OP-USD         5 -0.017686 -0.088431  0.200000
SAND-USD       2 -0.030000 -0.060000  0.000000
DOGE-USD       2 -0.030000 -0.060000  0.000000
SNX-USD       15 -0.003115 -0.046724  0.266667
KAVA-USD       8 -0.005328 -0.042626  0.250000
SOL-USD        1 -0.030000 -0.030000  0.000000
ALGO-USD       1 -0.030000 -0.030000  0.000000
NEO-USD        1 -0.030000 -0.030000  0.000000
BNB-USD        1 -0.030000 -0.030000  0.000000
CHZ-USD       12 -0.002131 -0.025575  0.250000
EOS-USD        5 -0.004218 -0.021091  0.200000
VET-USD        1 -0.020274 -0.020274  0.000000
AVAX-USD   

In [9]:
kept_no_top1 = [t for t in kept_tickers_dn if t not in top1_coins]
kept_no_top3 = [t for t in kept_tickers_dn if t not in top3_coins]

def subset_universe(tickers):
    p = prices_dn[tickers]
    v = volumes_dn[tickers]
    h = highs_dn[tickers]
    l = lows_dn[tickers]
    return p, v, h, l

p1, v1, h1, l1 = subset_universe(kept_no_top1)
p3, v3, h3, l3 = subset_universe(kept_no_top3)

test_no_top1 = run_backtest(p1, v1, frozen_daily, highs=h1, lows=l1, bar_freq='1D',
                             start_idx=split_idx_daily, label='test-no-top1')
test_no_top3 = run_backtest(p3, v3, frozen_daily, highs=h3, lows=l3, bar_freq='1D',
                             start_idx=split_idx_daily, label='test-no-top3')

print(f"Baseline test (all {len(kept_tickers_dn)} coins):        "
      f"Trades={test_daily['n_trades']}, Gross Sharpe={test_daily['gross_sharpe']}, Net Sharpe={test_daily['net_sharpe']}")
print(f"Excluding top 1 ({top1_coins[0]}):            "
      f"Trades={test_no_top1['n_trades']}, Gross Sharpe={test_no_top1['gross_sharpe']}, Net Sharpe={test_no_top1['net_sharpe']}")
print(f"Excluding top 3 ({top3_coins}): "
      f"Trades={test_no_top3['n_trades']}, Gross Sharpe={test_no_top3['gross_sharpe']}, Net Sharpe={test_no_top3['net_sharpe']}")
print("\nSame check on the regime-filtered design:")
p1r, v1r, h1r, l1r = subset_universe(kept_no_top1)
test_regime_no_top1 = run_backtest(p1r, v1r, frozen_regime, highs=h1r, lows=l1r, bar_freq='1D',
                                    start_idx=split_idx_daily, label='test-regime-no-top1')
p3r, v3r, h3r, l3r = subset_universe(kept_no_top3)
test_regime_no_top3 = run_backtest(p3r, v3r, frozen_regime, highs=h3r, lows=l3r, bar_freq='1D',
                                    start_idx=split_idx_daily, label='test-regime-no-top3')
print(f"Regime-filtered, excluding top 1: Trades={test_regime_no_top1['n_trades']}, "
      f"Gross Sharpe={test_regime_no_top1['gross_sharpe']}, Net Sharpe={test_regime_no_top1['net_sharpe']}")
print(f"Regime-filtered, excluding top 3: Trades={test_regime_no_top3['n_trades']}, "
      f"Gross Sharpe={test_regime_no_top3['gross_sharpe']}, Net Sharpe={test_regime_no_top3['net_sharpe']}")


Baseline test (all 45 coins):        Trades=723, Gross Sharpe=2.462, Net Sharpe=1.392
Excluding top 1 (SHIB-USD):            Trades=699, Gross Sharpe=1.292, Net Sharpe=0.237
Excluding top 3 (['SHIB-USD', 'MKR-USD', 'ZEC-USD']): Trades=654, Gross Sharpe=-0.275, Net Sharpe=-1.461

Same check on the regime-filtered design:
Regime-filtered, excluding top 1: Trades=470, Gross Sharpe=0.709, Net Sharpe=-0.164
Regime-filtered, excluding top 3: Trades=428, Gross Sharpe=-0.695, Net Sharpe=-1.657


In [10]:
import statsmodels.api as sm

full_result_daily = run_backtest(prices_dn, volumes_dn, frozen_daily, highs=highs_dn, lows=lows_dn,
                                  bar_freq='1D', label='full-sample')
strat_daily = full_result_daily['net_daily']
btc_daily = np.log(prices_dn['BTC-USD'] / prices_dn['BTC-USD'].shift(1)).reindex(strat_daily.index).fillna(0)

X = sm.add_constant(btc_daily)
model = sm.OLS(strat_daily, X).fit()

print(f"Annualized Alpha:   {model.params['const'] * 365 * 100:.2f}%")
print(f"Alpha t-stat:       {model.tvalues['const']:.2f}")
print(f"BTC Beta:           {model.params['BTC-USD']:.3f}")
print(f"Correlation to BTC: {strat_daily.corr(btc_daily):.3f}")
print("\nFull Regression Summary:")
print(model.summary())


Annualized Alpha:   93.03%
Alpha t-stat:       0.89
BTC Beta:           -0.024
Correlation to BTC: -0.005

Full Regression Summary:
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                   0.04335
Date:                Wed, 23 Sep 2026   Prob (F-statistic):              0.835
Time:                        13:53:49   Log-Likelihood:                 1163.4
No. Observations:                1459   AIC:                            -2323.
Df Residuals:                    1457   BIC:                            -2312.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]